# Amazon US BSR by Zinus Collection 

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import re
import math
from datetime import date

In [2]:
import os
import sys
import io
from io import BytesIO
import csv
import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [3]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../market-analysis-project-91130-5213911f50a5.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [9]:
sql = f"""
select * from wook.mart_amz_bsr_by_collection
where rank <= 20;
"""

df_bsr = bqclient.query(sql).to_dataframe()

In [11]:
print(df_bsr)

                 last_category  rank        asin    bsr_date   yr_qt bsr_year  \
0                       Tables    11  B000Y1W8GE  2024-02-24  24Y_1Q     2024   
1                       Tables    19  B000Y1W8GE  2024-02-26  24Y_1Q     2024   
2                       Tables    19  B002BTYRF2  2024-01-06  24Y_1Q     2024   
3                       Tables    12  B002BTYRF2  2024-01-02  24Y_1Q     2024   
4       Living Room Table Sets    14  B002O8COSM  2024-01-29  24Y_1Q     2024   
...                        ...   ...         ...         ...     ...      ...   
192595         Sofas & Couches    15  B0G528R114  2025-12-29  25Y_4Q     2025   
192596         Sofas & Couches    18  B0G528R114  2025-12-30  25Y_4Q     2025   
192597         Sofas & Couches    17  B0G528R114  2025-12-31  25Y_4Q     2025   
192598         Sofas & Couches    11  B0G528R114  2025-12-13  25Y_4Q     2025   
192599         Sofas & Couches    14  B0G528R114  2025-12-16  25Y_4Q     2025   

                           

In [11]:
df_bsr.duplicated().sum()

0

In [13]:
df1 = df_bsr.copy()

df1['ctgry_type'] = df1['last_category'].apply(
    lambda x: 'MATT' if x == 'Matttresses' else 'NON-MATT'
)
df1['bsr_date'] = pd.to_datetime(df1['bsr_date'], errors='coerce')
df1['year'] = df1['bsr_date'].dt.year
print(df1)

                 last_category  rank        asin   bsr_date   yr_qt bsr_year  \
0                       Tables    11  B000Y1W8GE 2024-02-24  24Y_1Q     2024   
1                       Tables    19  B000Y1W8GE 2024-02-26  24Y_1Q     2024   
2                       Tables    19  B002BTYRF2 2024-01-06  24Y_1Q     2024   
3                       Tables    12  B002BTYRF2 2024-01-02  24Y_1Q     2024   
4       Living Room Table Sets    14  B002O8COSM 2024-01-29  24Y_1Q     2024   
...                        ...   ...         ...        ...     ...      ...   
192595         Sofas & Couches    15  B0G528R114 2025-12-29  25Y_4Q     2025   
192596         Sofas & Couches    18  B0G528R114 2025-12-30  25Y_4Q     2025   
192597         Sofas & Couches    17  B0G528R114 2025-12-31  25Y_4Q     2025   
192598         Sofas & Couches    11  B0G528R114 2025-12-13  25Y_4Q     2025   
192599         Sofas & Couches    14  B0G528R114 2025-12-16  25Y_4Q     2025   

                             brand coll

In [15]:
def get_brand_share_by_category(
    df,
    category,
    years=[2025],
    category_col='last_category',
    brand_col='brand',
    top_n=10,
    exclude_others=True
):
    """
    특정 category(bsr_ctgry) 기준으로
    - brand BSR 노출 수
    - category 내 brand share 계산
    - Top N 브랜드 반환
    """

    # === 1. 연도 + 카테고리 필터 ===
    df_use = df[
        (df['year'].isin(years)) &
        (df[category_col] == category)
    ].copy()

    if df_use.empty:
        print(f"[INFO] No data for category='{category}', years={years}")
        return pd.DataFrame()

    # === 2. (category, brand) BSR 노출 횟수 ===
    brand_cnt = (
        df_use
        .groupby(brand_col)
        .size()
        .reset_index(name='bsr_cnt')
    )

    # === 3. category 기준 share 계산 ===
    brand_share = (
        brand_cnt
        .assign(
            cat_total_cnt=lambda x: x['bsr_cnt'].sum(),
            bsr_share=lambda x: x['bsr_cnt'] / x['cat_total_cnt'],
            bsr_share_pct=lambda x: x['bsr_share'] * 100
        )
        .sort_values('bsr_share', ascending=False)
    )
    #print(brand_share)

    # === 5. Top N ===
    return brand_share.head(top_n)

In [17]:
categories = ['Beds', 'Bed Frames', 'Mattresses']

top10_base_year = {
    cat: get_brand_share_by_category(df1, category=cat, years=[2025])
    for cat in categories
}

In [19]:
top10_base_year['Bed Frames']

,brand,bsr_cnt,cat_total_cnt,bsr_share,bsr_share_pct
0,ALLEWIE,857,7300,0.117397,11.739726
1,AMAZON BASICS,733,7300,0.100411,10.041096
47,ZINUS,487,7300,0.066712,6.671233
45,YAHEETECH,484,7300,0.066301,6.630137
14,FURNULEM,433,7300,0.059315,5.931507
24,LIFEZONE,366,7300,0.050137,5.013699
32,MELLOW,365,7300,0.050000,5.000000
15,HAFENPO,365,7300,0.050000,5.000000
26,LIKIMIO,365,7300,0.050000,5.000000
33,NEW JETO,365,7300,0.050000,5.000000


In [33]:
# 분기별 집계하기 
brand_cnt = (
    df1
    .groupby(['last_category', 'yr_qt', 'brand'])
    .size()
    .reset_index(name='bsr_cnt')
)

brand_cnt[brand_cnt['last_category']=='Bed Frames']

,last_category,yr_qt,brand,bsr_cnt
54,Bed Frames,24Y_1Q,ALAZYHOME,8
55,Bed Frames,24Y_1Q,ALDRICH,6
56,Bed Frames,24Y_1Q,ALLEWIE,66
57,Bed Frames,24Y_1Q,AMAZON BASICS,167
58,Bed Frames,24Y_1Q,AVENCO,2
...,...,...,...,...
340,Bed Frames,25Y_4Q,SWEETCRISPY,54
341,Bed Frames,25Y_4Q,VECELO,22
342,Bed Frames,25Y_4Q,YAHEETECH,168
343,Bed Frames,25Y_4Q,ZINUS,38


In [69]:
qt_total = (
    brand_cnt
    .groupby(['last_category','yr_qt'], as_index=False)
    .agg(total=('bsr_cnt','sum'))
)

qt_total[qt_total['last_category']=='Mattresses']

,last_category,yr_qt,total
74,Mattresses,24Y_1Q,1820
75,Mattresses,24Y_2Q,1820
76,Mattresses,24Y_3Q,1840
77,Mattresses,24Y_4Q,1840
78,Mattresses,25Y_1Q,1800
79,Mattresses,25Y_2Q,1820
80,Mattresses,25Y_3Q,1839
81,Mattresses,25Y_4Q,1840


In [67]:
qt_cnt = (
    df1
    .groupby(['last_category', 'yr_qt'])
    .size()
    .reset_index(name='qt_cnt')
)
qt_cnt

,last_category,yr_qt,qt_cnt
0,Adjustable Bases,25Y_3Q,700
1,Adjustable Bases,25Y_4Q,1840
2,Bed Frames,24Y_1Q,1820
3,Bed Frames,24Y_2Q,1820
4,Bed Frames,24Y_3Q,1840
...,...,...,...
101,Television Stands & Entertainment Centers,24Y_4Q,1840
102,Television Stands & Entertainment Centers,25Y_1Q,1800
103,Television Stands & Entertainment Centers,25Y_2Q,1820
104,Television Stands & Entertainment Centers,25Y_3Q,1840


In [61]:
# 분기별 집계하기 
zinus_cnt = (
    df1.loc[df1['brand']=='ZINUS']
    .groupby(['last_category', 'yr_qt', 'collection_adj'])
    .size()
    .reset_index(name='bsr_cnt')
)
zinus_cnt
#zinus_cnt[zinus_cnt['last_category']=='Bed Frames']

,last_category,yr_qt,collection_adj,bsr_cnt
0,Bed Frames,24Y_1Q,Michelle,75
1,Bed Frames,24Y_1Q,Shawn,8
2,Bed Frames,24Y_1Q,Van,8
3,Bed Frames,24Y_2Q,Dawn,3
4,Bed Frames,24Y_2Q,Kai,19
...,...,...,...,...
264,Tables,24Y_2Q,Becky,13
265,Tables,24Y_2Q,Juliet,1
266,Tables,24Y_3Q,Jen,11
267,Tables,24Y_4Q,Jen,14


In [59]:
# 3) 결합
result = (
    zinus_cnt
    .merge(qt_total, on=['last_category', 'yr_qt'], how='left')
)

print(result)

    last_category   yr_qt collection_adj  bsr_cnt  total
0      Bed Frames  24Y_1Q       Michelle       75   1820
1      Bed Frames  24Y_1Q          Shawn        8   1820
2      Bed Frames  24Y_1Q            Van        8   1820
3      Bed Frames  24Y_2Q           Dawn        3   1820
4      Bed Frames  24Y_2Q            Kai       19   1820
..            ...     ...            ...      ...    ...
264        Tables  24Y_2Q          Becky       13   1818
265        Tables  24Y_2Q         Juliet        1   1818
266        Tables  24Y_3Q            Jen       11   1840
267        Tables  24Y_4Q            Jen       14   1840
268        Tables  25Y_2Q            Jen        1   1820

[269 rows x 5 columns]
